# Phần 0 — Quy trình xử lý dữ liệu: từ dữ liệu thô đến dữ liệu sạch

Notebook này **đã được chạy sẵn**: mọi bảng ví dụ và con số bên dưới được lưu trong file, nên có thể xem trực tiếp trên
GitHub **mà không cần tải dữ liệu** (MagnetDB ~117 GB). Muốn chạy lại, làm theo mục *Tải và lưu dữ liệu* trong `README.md`.

Mỗi bước gồm: **đầu vào → việc làm → ví dụ dữ liệu → số lượng còn lại**. Mã nguồn đầy đủ của từng bước nằm trong `src/`.

```
 DỮ LIỆU THÔ                         TIỀN XỬ LÝ / TÍCH HỢP                               DỮ LIỆU SẠCH
 ───────────                         ─────────────────────                               ────────────
 MagnetDB (SQLite 117 GB) ──► 01–02  file video + thời điểm phát hiện ──┐
                          └─► 04     tự ghép TÊN torrent → phim IMDb ───┤
 IMDb (TSV)               ──► 03     danh mục phim + chỉ mục tên (akas) ┼──► 05 danh sách phim ──► 09 LÀM SẠCH ──► torrents_clean
 TMDb API (JSON)          ──► 06–07  ngày phát hành, tài chính, streaming┘                         (bộ lọc A–J)   movie_analysis
                              08     đánh giá chất lượng ghép
 MagnetDB (toàn bộ)       ──► 10     tổng quan hệ sinh thái (mọi loại nội dung) ──────────────────────────────► mdb_*
```

In [1]:
import sys, json
from pathlib import Path
import duckdb, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "viz_style.py").exists())
DC = ROOT / "data_clean"
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)
con = duckdb.connect(); con.sql("SET enable_progress_bar = false")
q = lambda sql: con.sql(sql.replace("@", (DC.as_posix() + "/"))).df()   # @x.parquet = data_clean/x.parquet
n = lambda sql: con.sql(sql.replace("@", (DC.as_posix() + "/"))).fetchone()[0]

## Bước 0 — Dữ liệu thô: ba nguồn, ba định dạng, không có khóa chung

| Nguồn | Định dạng | Quy mô | Khóa phim |
|---|---|---|---|
| **MagnetDB** | SQLite 117 GB: bảng `torrents` (28,6 triệu dòng), `files` (950 triệu dòng) | lớn nhất | **không có** — chỉ có tên torrent/tên file tự do |
| **IMDb** | 3 file TSV nén: `title.basics`, `title.ratings`, `title.akas` | ~760 nghìn phim | `tconst` (vd. `tt1630029`) |
| **TMDb** | API trả JSON, 1 file/phim | ~142 nghìn phim | `imdb_id` → nối thẳng với IMDb |

Ví dụ một dòng **torrent thô** của MagnetDB — tên là chuỗi tự do, phải tự tách ra phim nào:

In [2]:
q("""SELECT torrent_id, name AS ten_torrent, round(total_size/1e9, 2) AS GB, discovered_at_utc AS thoi_diem_phat_hien
     FROM '@mdb_torrents_all.parquet'
     WHERE name ILIKE '%Avatar.The.Way.of.Water.2022%' OR name ILIKE '%Top.Gun.Maverick.2022%' OR name ILIKE '%Dune.2021.1080p%'
     ORDER BY torrent_id LIMIT 5""")

,torrent_id,ten_torrent,GB,thoi_diem_phat_hien
0,16813518,Dune.2021.1080p.HDCAM-C1NEM4[TGx],3.60,2021-09-19 12:52:39
1,16860821,Dune.2021.1080p BluRay x264 DTS-HD Regraded,0.70,2021-09-22 17:51:45
2,17196933,Dune.2021.1080p.HDRip.X264-EVO[TGx],4.71,2021-10-17 04:58:05
3,17198468,Dune.2021.1080p.HDRip.1600MB.DD5.1.x264-GalaxyRG[TGx],1.72,2021-10-17 07:32:34
4,17199043,Dune.2021.1080p.HDRip.X264.AC3-EVO,11.39,2021-10-17 08:29:13


Tổng quy mô bảng `torrents` (mọi loại nội dung: phim, nhạc, phần mềm, ảnh…) và phần được đưa vào bước tự ghép tên (Bước 4):
chỉ giữ torrent **phát hiện từ 01/04/2019 đến trước 30/09/2024** và **tổng dung lượng ≥ 100 MB** (một bộ phim hiếm khi nhỏ hơn;
bỏ bớt nhạc lẻ, ảnh, phụ đề…). Đây là lý do Bước 4 làm việc trên ~23,5 triệu torrent chứ không phải 28,6 triệu:

In [3]:
# Đầu vào của Bước 4 (src/04): cùng khoảng thời gian và ngưỡng dung lượng
DAU_VAO = """SELECT torrent_id FROM '@mdb_torrents_all.parquet'
            WHERE discovered_at_utc >= '2019-04-01' AND discovered_at_utc < '2024-09-30' AND total_size >= 100e6"""
q("""SELECT count(*) AS so_torrent, min(discovered_at_utc) AS tu, max(discovered_at_utc) AS den,
            count(*) FILTER (WHERE discovered_at_utc >= '2019-04-01' AND discovered_at_utc < '2024-09-30') AS trong_04_2019_09_2024,
            count(*) FILTER (WHERE discovered_at_utc >= '2019-04-01' AND discovered_at_utc < '2024-09-30' AND total_size >= 100e6) AS dau_vao_buoc_4
     FROM '@mdb_torrents_all.parquet'""")

,so_torrent,tu,den,trong_04_2019_09_2024,dau_vao_buoc_4
0,28606694,2019-01-01 21:19:20,2024-10-04 19:58:11,27576351,23468587


## Bước 1–2 — Mã IMDb có sẵn trong MagnetDB (`src/01`, `src/02`)

**Đầu vào:** `matchedVideos_sample.csv` (MagnetDB đã tự ghép file video với IMDb, chỉ giữ điểm khớp `matchScore > 138`) +
bảng `torrents` trong SQLite (lấy thời điểm phát hiện).
**Việc làm:** chuyển CSV → Parquet, lấy `discovered_on` của từng torrent, giải mã lại ~1.300 tên torrent bị lỗi mã hóa
(Windows-1251, tiếng Nga).
**Vấn đề phát hiện:** mã ghép sẵn **có lỗi** và **phủ rất thấp**. Ví dụ các ca gán sai rõ ràng:

In [4]:
q("""SELECT path AS ten_file, parsed_title AS ten_tach_duoc, match_title AS MagnetDB_gan_vao, round(match_score) AS diem
     FROM '@matched_files.parquet'
     WHERE path ILIKE '1. Introduction%' OR path ILIKE '%Girlfriend.Experience.S0%' OR path ILIKE 'Amarcord%'
     QUALIFY row_number() OVER (PARTITION BY match_title ORDER BY file_id) = 1 LIMIT 5""")

,ten_file,ten_tach_duoc,MagnetDB_gan_vao,diem
0,1. Introduction/13. If Statements.mp4,1. Introduction/13. If Statements.,Introduction to Statistics,161.0
1,1. Introduction/26. Generators.mp4,1. Introduction/26. Generators.,O Intrépido General Custer,138.0
2,the.girlfriend.experience.s02e05.repack.720p.web.h264-convoy.mkv,the girlfriend experience,The Girl Expert,154.0
3,1. Introduction/1. Introduction.mp4,1. Introduction/1. Introduction.,The intruder (El intruso),238.0
4,1. Introduction/6. JVM Architecture.mp4,1. Introduction/6. JVM Architecture.,An Introduction to Architecture,175.0


In [5]:
q("""SELECT (SELECT count(*) FROM '@matched_files.parquet') AS so_file_video_da_ghep,
            (SELECT count(DISTINCT torrent_id) FROM '@matched_files.parquet') AS so_torrent,
            (SELECT count(DISTINCT imdb_id) FROM '@matched_files.parquet') AS so_ma_IMDb""")

,so_file_video_da_ghep,so_torrent,so_ma_IMDb
0,1562572,540329,78740


## Bước 3 — Danh mục IMDb và chỉ mục tên (`src/03`)

**Việc làm:** giữ phim (`titleType = movie`), gắn điểm/số vote, và tạo **chỉ mục tên chuẩn hóa**: mỗi phim có nhiều tên
(tên gốc, tên tiếng Anh, tên ở từng nước trong `title.akas`), mọi tên được **bỏ dấu, viết thường, bỏ ký tự đặc biệt**.
Nhờ đó tên torrent tiếng Nga, Ý, Tây Ban Nha… vẫn tra được. Ví dụ các tên của *The Big Lebowski* trong chỉ mục:

In [6]:
q("SELECT norm AS ten_chuan_hoa, tconst, start_year, num_votes FROM '@imdb_title_index.parquet' WHERE tconst = 'tt0118715' LIMIT 12")

,ten_chuan_hoa,tconst,start_year,num_votes
0,谋杀绿脚趾,tt0118715,1998,925652
1,ba tuoc lebowski,tt0118715,1998,925652
2,den store lebowski,tt0118715,1998,925652
3,велики лебовски,tt0118715,1998,925652
4,lebowski bozorg,tt0118715,1998,925652
5,el gran lebowski,tt0118715,1998,925652
6,a nagy lebowski,tt0118715,1998,925652
7,ο μεγαλος λεμποφσκι,tt0118715,1998,925652
8,великии лебовські,tt0118715,1998,925652
9,veliki lebowski,tt0118715,1998,925652


In [7]:
q("""SELECT (SELECT count(*) FROM '@imdb_movies_all.parquet') AS moi_phim_IMDb,
            (SELECT count(*) FROM '@imdb_catalog.parquet') AS ung_vien_co_rating,
            (SELECT count(*) FROM '@imdb_title_index.parquet') AS so_ten_trong_chi_muc""")

,moi_phim_IMDb,ung_vien_co_rating,so_ten_trong_chi_muc
0,758524,467236,1875336


## Bước 4 — Tự ghép tên torrent → phim (`src/04`, bước tích hợp quan trọng nhất)

**Vì sao:** MagnetDB tách tên đúng nhưng tra IMDb sai (vd. `Deadpool.2016.1080p.BluRay` → một video ca nhạc, điểm 11 nên bị loại),
nên bom tấn gần như vắng mặt trong dữ liệu gốc.
**Việc làm (toàn bộ bằng SQL trên 23,5 triệu torrent ≥ 100 MB phát hiện 04/2019–09/2024 — xem Bước 0; ~2 phút):**
1. Bỏ tiền tố rác (`[YTS.MX]`, `www.site.com -`), đuôi file.
2. Tách **NĂM** (1920–2029, không nhầm với `1080p`, `1920x1080`) và **TÊN** = phần trước năm.
3. Bỏ torrent phim bộ (`S01E02`, `Season`, `Сезон`…).
4. Tra **chính xác** tên chuẩn hóa trong chỉ mục IMDb, năm lệch ≤ 1, chỉ phim lẻ.
5. Đánh dấu **mơ hồ** khi trùng tên nhiều phim, hoặc tên quá chung chung của phim ít người biết ("New", "Cricket").

Ví dụ kết quả ghép:

In [8]:
q("""SELECT n.torrent_name AS ten_torrent, n.name_title AS ten_tach, n.name_year AS nam, m.title AS phim_IMDb, m.start_year
     FROM '@name_matches.parquet' n JOIN '@movies.parquet' m USING (imdb_id)
     WHERE NOT n.ambiguous AND m.num_votes > 100000
       AND (n.torrent_name ILIKE '%Банды.Нью-Йорка%' OR n.torrent_name ILIKE '%I Guerrieri (1970)%'
            OR n.torrent_name ILIKE 'Parasite.2019.PL%' OR n.torrent_name ILIKE '%Scarface.1983.1080p.BluRay.10bit%'
            OR n.torrent_name ILIKE '%Drácula de Bram Stoker%')
     QUALIFY row_number() OVER (PARTITION BY m.imdb_id ORDER BY n.torrent_id) = 1""")

,ten_torrent,ten_tach,nam,phim_IMDb,start_year
0,Drácula de Bram Stoker 1992 WWW.BLUDV.COM,Drácula de Bram Stoker,1992,Dracula,1992
1,Parasite.2019.PLSUBBED.480p.WEB-DL.XViD.AC3-LLA.avi,Parasite,2019,Parasite,2019
2,Scarface.1983.1080p.BluRay.10bit.x265-HazMatt.mkv,Scarface,1983,Scarface,1983
3,Банды.Нью-Йорка.Gangs.of.New.York.2002.US.Award-Winning.Collection.BDRip-HEVC.1080p.mkv,Банды.Нью-Йорка.Gangs.of.New.York,2002,Gangs of New York,2002


In [9]:
q("""SELECT count(*) AS torrent_ghep_duoc, count(*) FILTER (WHERE ambiguous) AS ca_mo_ho_bi_bo,
            count(DISTINCT imdb_id) FILTER (WHERE NOT ambiguous) AS so_phim
     FROM '@name_matches.parquet'""")

,torrent_ghep_duoc,ca_mo_ho_bi_bo,so_phim
0,2253337,193739,122990


## Bước 5–7 — Danh sách phim và dữ liệu TMDb (`src/05`, `src/06`, `src/07`)

**Việc làm:** hợp danh sách phim của hai cách ghép (bỏ mục không phải phim như sự kiện thể thao); gọi TMDb API cho từng phim
(và mọi phim IMDb ≥ 1.000 vote) → JSON được lưu đệm → chuyển thành 3 bảng: thông tin phim, ngày phát hành theo nước/loại,
nơi xem streaming theo nước. Ví dụ dòng đã chuẩn hóa:

In [10]:
q("""SELECT imdb_id, tmdb_title, release_date::DATE AS ngay_phat_hanh, budget, revenue,
            first_theatrical::DATE AS ra_rap_som_nhat, first_digital::DATE AS ban_online_som_nhat, us_streaming_providers
     FROM '@tmdb_movies.parquet' WHERE imdb_id IN ('tt1630029','tt1745960','tt1160419')""")

,imdb_id,tmdb_title,ngay_phat_hanh,budget,revenue,ra_rap_som_nhat,ban_online_som_nhat,us_streaming_providers
0,tt1160419,Dune,2021-09-15,165000000.0,4.106685e+08,2021-09-15,2021-10-21,"HBO Max,HBO Max Amazon Channel,TBS,TNT,YouTube TV,tru TV"
1,tt1630029,Avatar: The Way of Water,2022-12-14,460000000.0,2.334485e+09,2022-12-14,2023-03-28,Disney Plus
2,tt1745960,Top Gun: Maverick,2022-05-21,170000000.0,1.488733e+09,2022-05-21,2022-08-21,"Netflix,Netflix Standard with Ads,Paramount Plus Essential,Paramount Plus Premium,Para..."


In [11]:
q("SELECT imdb_id, country AS nuoc, type AS loai, release_date::DATE AS ngay FROM '@tmdb_release_dates.parquet' WHERE imdb_id = 'tt1745960' ORDER BY ngay LIMIT 8")

,imdb_id,nuoc,loai,ngay
0,tt1745960,US,1,2022-05-04
1,tt1745960,MX,1,2022-05-06
2,tt1745960,FR,1,2022-05-18
3,tt1745960,GB,1,2022-05-19
4,tt1745960,CO,3,2022-05-21
5,tt1745960,MX,2,2022-05-21
6,tt1745960,UY,2,2022-05-21
7,tt1745960,MA,3,2022-05-24


Loại phát hành TMDb: 1 công chiếu, 2 rạp giới hạn, 3 rạp, **4 Digital (bản online)**, 5 đĩa, 6 TV.

## Bước 8 — Kiểm tra thuật toán ghép tên của Bước 4

Không có đáp án đúng cho 23,5 triệu torrent, nên thuật toán ghép tên ở Bước 4 được **so với nhãn có sẵn của MagnetDB**
(mã IMDb mà MagnetDB tự gán cho file video, Bước 1–2) trên những torrent mà cả hai cùng xét: thuộc đầu vào Bước 4 và
được MagnetDB gán vào một phim lẻ.

- **Mức đồng thuận** = số torrent thuật toán ghép trùng nhãn MagnetDB ÷ số torrent thuật toán đưa ra kết quả (không mơ hồ).
  Cho biết: *khi thuật toán đã trả lời, bao nhiêu lần giống MagnetDB.*
- **Độ phủ** = số torrent ghép trùng nhãn MagnetDB ÷ mọi torrent có nhãn MagnetDB.
  Thấp hơn vì thuật toán **cố ý từ chối** tên không có năm hoặc tên mơ hồ thay vì đoán.

Nhãn MagnetDB **cũng sai** (Bước 1–2), nên đây là *mức đồng thuận*, không phải độ chính xác tuyệt đối:

In [12]:
con.execute(("""CREATE OR REPLACE TEMP TABLE kiem_tra AS
WITH nhan AS (   -- nhãn MagnetDB: torrent có file video được MagnetDB gán vào một phim lẻ IMDb
    SELECT f.torrent_id, list(DISTINCT f.imdb_id) AS ma_magnetdb, mode(f.imdb_id) AS ma_magnetdb_chinh
    FROM '@matched_files.parquet' f JOIN '@imdb_movies_all.parquet' m ON m.tconst = f.imdb_id
    WHERE f.torrent_id IN (""" + DAU_VAO + """)
    GROUP BY 1)
SELECT nhan.*, o.imdb_id AS ma_thuat_toan, o.torrent_name,
       coalesce(NOT o.ambiguous, false) AS tra_loi,
       coalesce(NOT o.ambiguous AND list_contains(nhan.ma_magnetdb, o.imdb_id), false) AS trung
FROM nhan LEFT JOIN '@name_matches.parquet' o USING (torrent_id)""").replace("@", DC.as_posix() + "/"))
display(q("""SELECT count(*) AS torrent_co_nhan_MagnetDB, count(*) FILTER (WHERE tra_loi) AS thuat_toan_tra_loi,
            count(*) FILTER (WHERE trung) AS trung_nhan_MagnetDB,
            round(100.0 * count(*) FILTER (WHERE trung) / count(*) FILTER (WHERE tra_loi), 1) AS muc_dong_thuan_pct,
            round(100.0 * count(*) FILTER (WHERE trung) / count(*), 1) AS do_phu_pct
     FROM kiem_tra"""))
# Ví dụ ca bất đồng: phần lớn là MagnetDB gán nhầm phần phim hoặc bản khác năm
q("""SELECT k.torrent_name AS ten_torrent, a.title || ' (' || a.start_year || ')' AS thuat_toan_ghep,
            b.title || ' (' || b.start_year || ')' AS MagnetDB_gan
     FROM kiem_tra k JOIN '@imdb_movies_all.parquet' a ON a.tconst = k.ma_thuat_toan
                     JOIN '@imdb_movies_all.parquet' b ON b.tconst = k.ma_magnetdb_chinh
     WHERE k.tra_loi AND NOT k.trung
       AND (k.torrent_name ILIKE 'Paranormal Activity 4 [BluRay%' OR k.torrent_name ILIKE 'Perfetti.Sconosciuti.2016.%'
            OR k.torrent_name ILIKE 'The.Kindergarten.Teacher.2018.%' OR k.torrent_name ILIKE 'The Neverending Story 1984%'
            OR k.torrent_name ILIKE 'Блуждающая Земля.2019%')
     QUALIFY row_number() OVER (PARTITION BY a.tconst ORDER BY k.torrent_id) = 1
     ORDER BY a.start_year""")

,torrent_co_nhan_MagnetDB,thuat_toan_tra_loi,trung_nhan_MagnetDB,muc_dong_thuan_pct,do_phu_pct
0,226006,129184,118812,92.0,52.6


,ten_torrent,thuat_toan_ghep,MagnetDB_gan
0,The Neverending Story 1984.J.720p. BRRip.BG Audio.MadMax & Goredolo,The NeverEnding Story (1984),The NeverEnding Story II: The Next Chapter (1990)
1,Paranormal Activity 4 [BluRay 1080p][AC3 5.1 Castellano_DTS English + Subs. ES_EN][2012],Paranormal Activity 4 (2012),Paranormal Activity (2007)
2,Perfetti.Sconosciuti.2016.PL.720p.BDRip.XviD.AC3-inTGrity,Perfect Strangers (2016),Perfect Strangers (2022)
3,The.Kindergarten.Teacher.2018.BRRip.x264.AAC-HUD,The Kindergarten Teacher (2018),The Kindergarten Teacher (2014)
4,Блуждающая Земля.2019.(RUS).WEB-DLRip by ExKinoRay & Shkiper.avi,The Wandering Earth (2019),The Wandering Earth II (2023)


## Bước 9 — Làm sạch và gộp hai nguồn (`src/09`, bộ lọc A–J)

Đây là bước biến dữ liệu "ghép được" thành dữ liệu **tin được**. Bảng dưới là nhật ký số lượng còn lại sau từng bộ lọc
(`data_clean/cleaning_log.csv`):

In [13]:
log = pd.read_csv(DC / "cleaning_log.csv")
log.rename(columns={"source": "nguon", "step": "buoc", "files": "so_file", "torrents": "so_torrent", "movies": "so_phim"})

,nguon,buoc,so_file,so_torrent,so_phim
0,magnetdb,0. slice 2019-04 -> 2024-09 (mọi loại),1504330.0,517557,77474
1,magnetdb,A. chỉ PHIM (titleType=movie),335418.0,226484,33203
2,magnetdb,B. bỏ file có số mùa/tập,278933.0,216009,32646
3,magnetdb,C. bỏ file < 100MB,235799.0,192784,30772
4,magnetdb,D. năm trong tên file khớp năm phim (±1),206683.0,174976,29357
5,magnetdb,E. tên file giống tên phim >= 60,199230.0,171763,28451
6,magnetdb,E2. số phần phim khớp (Part 2 ≠ Part 1),194085.0,170163,28407
7,magnetdb,"-> cặp (torrent, phim)",NaN,170163,28407
8,name_match,0. ghép được (tên + năm khớp IMDb),NaN,2253337,133388
9,name_match,bỏ ca mơ hồ (>1 phim trùng tên ±1 năm),NaN,2059598,122990


Ý nghĩa từng bộ lọc và **ví dụ thật của dòng bị loại**:

| Bộ lọc | Loại bỏ cái gì |
|---|---|
| A | Không phải phim lẻ (phim bộ, video, chương trình TV) |
| B | Tập phim bộ bị gán nhầm vào phim lẻ cùng tên (`S01E05`) |
| C | Tệp < 100 MB: trailer, clip, sample |
| D | Năm trong tên file lệch > 1 năm so với năm phim |
| E / E2 | Tên file không giống bất kỳ tên nào của phim; sai phần phim (Part 2 ≠ Part 1) |
| F | Torrent xuất hiện sớm hơn 60 ngày trước lần ra mắt đầu tiên của phim |
| G | Hai nguồn ghép mâu thuẫn (torrent 1 phim, hai cách ghép ra hai phim khác nhau) |
| H, I | Tên torrent có từ khóa phim bộ; năm trên tên torrent lệch |
| J | **Torrent giả**: tự xưng bản WEB/BluRay khi phim chưa có bản đó |

**Ví dụ bộ lọc B và D** — tập phim bộ và năm lệch bị MagnetDB gán vào phim lẻ:

In [14]:
q("""SELECT s.path AS ten_file, s.parsed_year AS nam_trong_ten, m.title AS bi_gan_vao, m.start_year AS nam_phim,
            CASE WHEN s.is_tv_hint THEN 'B: tập phim bộ' ELSE 'D: năm lệch' END AS bi_loai_vi
     FROM '@magnetdb_matched_slice.parquet' s JOIN '@movies.parquet' m USING (imdb_id)
     WHERE (s.is_tv_hint AND s.path ILIKE '%Series.of.Unfortunate.Events.S0%')
        OR (NOT s.is_tv_hint AND s.file_size >= 1e8 AND abs(s.parsed_year - m.start_year) > 1 AND m.title IN ('Gang of Roses', 'Independence Day'))
     QUALIFY row_number() OVER (PARTITION BY m.title ORDER BY s.file_id) <= 2""")

,ten_file,nam_trong_ten,bi_gan_vao,nam_phim,bi_loai_vi
0,01.Великолепная Пятерка.(2019).WEBRip.(AVC).by.Серый1779.Files-x.mkv,2019,Gang of Roses,2003,D: năm lệch
1,02.Великолепная Пятерка.(2019).WEBRip.(AVC).by.Серый1779.Files-x.mkv,2019,Gang of Roses,2003,D: năm lệch
2,День независимости.Возрождение.2016-1080p-HEVC.mkv,2016,Independence Day,1996,D: năm lệch
3,День независимости. 2016.avi,2016,Independence Day,1996,D: năm lệch
4,lemony.snickets.a.series.of.unfortunate.events.s02e03.proper.1080p.webrip.x264-webtifu...,<NA>,A Series of Unfortunate Events,2004,B: tập phim bộ
5,A.Series.of.Unfortunate.Events.S03E01.WEBRip.1080p.IdeaFilm.mkv,<NA>,A Series of Unfortunate Events,2004,B: tập phim bộ


**Ví dụ bộ lọc J** — torrent giả: bản "BluRay" của phim xuất hiện khi phim còn chưa ra rạp, cùng một dung lượng lặp lại:

In [15]:
q("""SELECT t.discovered_at_utc::DATE AS phat_hien, t.torrent_name AS ten_torrent, round(t.total_size/1e9, 2) AS GB,
            m.first_theatrical::DATE AS ra_rap, m.first_digital::DATE AS ban_online
     FROM '@torrents_clean.parquet' t JOIN '@movie_analysis.parquet' m USING (imdb_id)
     WHERE t.implausible AND m.title = 'Ant-Man and the Wasp: Quantumania'
     ORDER BY phat_hien LIMIT 6""")

,phat_hien,ten_torrent,GB,ra_rap,ban_online
0,2022-12-18,Ant-Man and the Wasp Quantumania 2023 BluRay X264,0.67,2023-02-15,2023-04-18
1,2022-12-23,Ant-Man and the Wasp Quantumania 2023 BluRay X265,0.67,2023-02-15,2023-04-18
2,2023-01-28,Ant-Man and the Wasp Quantumania (2023) BLURAY H264,0.67,2023-02-15,2023-04-18
3,2023-02-07,Ant-Man and the Wasp Quantumania 2023 BluRay X265,0.67,2023-02-15,2023-04-18
4,2023-02-12,Ant-Man and the Wasp Quantumania 2023 BluRay H264,0.67,2023-02-15,2023-04-18
5,2023-02-13,Ant-Man and the Wasp Quantumania.2023 BrRip X265,0.64,2023-02-15,2023-04-18


## Kết quả: hai bảng sạch dùng cho mọi phân tích

**`torrents_clean`** — 1 dòng = 1 cặp (torrent, phim), kèm nguồn ghép, chất lượng bản phát hành, cờ gói nhiều phim, cờ torrent giả:

In [16]:
q("""SELECT t.torrent_name, m.title AS phim, t.source AS nguon_ghep, t.quality_tier AS chat_luong, t.is_pack, t.implausible,
            t.discovered_at_utc::DATE AS phat_hien
     FROM '@torrents_clean.parquet' t JOIN '@movies.parquet' m USING (imdb_id)
     WHERE m.title IN ('Dune', 'Top Gun: Maverick') AND m.start_year IN (2021, 2022) AND t.in_stable_window AND NOT t.implausible AND t.quality_tier IN ('web', 'disc') AND regexp_matches(t.torrent_name, '^[A-Za-z0-9.:]+$')
     QUALIFY row_number() OVER (PARTITION BY m.title, t.source, t.quality_tier ORDER BY t.torrent_id) = 1""")

,torrent_name,phim,nguon_ghep,chat_luong,is_pack,implausible,phat_hien
0,Top.Gun.Maverick.2022.1080p.WEBRip.x264.AAC.mp4,Top Gun: Maverick,name_match,web,False,False,2022-08-17
1,Top.Gun.Maverick.2022.IMAX.BDRip.720p.ExKinoRay.mkv,Top Gun: Maverick,name_match,disc,False,False,2022-08-23
2,Dune.2021.D.BDRip.2.18Gb.MegaPeer,Dune,name_match,disc,False,False,2021-12-03
3,Dune.2021.1080p.HDRip.Dublado.mkv,Dune,name_match,web,False,False,2021-10-17


**`movie_analysis`** — 1 dòng = 1 phim: IMDb + TMDb + thống kê torrent trong giai đoạn crawler ổn định (04/2021 – 09/2024):

In [17]:
q("""SELECT title, start_year, num_votes, avg_rating, revenue, first_digital::DATE AS ban_online,
            n_torrents_stable AS so_torrent_on_dinh, n_torrents_stable_magnetdb AS chi_tinh_ma_MagnetDB
     FROM '@movie_analysis.parquet' ORDER BY n_torrents_stable DESC LIMIT 8""")

,title,start_year,num_votes,avg_rating,revenue,ban_online,so_torrent_on_dinh,chi_tinh_ma_MagnetDB
0,Avatar: The Way of Water,2022,649218,7.5,2.334485e+09,2023-03-28,1255,5
1,Spider-Man: No Way Home,2021,1078783,8.1,1.921426e+09,2022-03-14,952,2
2,Black Widow,2021,491007,6.6,3.797511e+08,2021-07-08,929,0
3,Dune,2021,1093705,8.0,4.106685e+08,2021-10-21,917,0
4,The Batman,2022,951822,7.8,7.727753e+08,2022-04-17,905,0
5,Doctor Strange in the Multiverse of Madness,2022,561969,6.8,9.557758e+08,2022-06-21,881,97
6,Top Gun: Maverick,2022,907995,8.2,1.488733e+09,2022-08-21,843,0
7,Eternals,2021,438002,6.2,4.020649e+08,2022-01-12,814,0


In [18]:
q("""SELECT count(*) AS cap_torrent_phim_sach, count(DISTINCT torrent_id) AS so_torrent, count(DISTINCT imdb_id) AS so_phim,
            count(*) FILTER (WHERE in_stable_window) AS trong_giai_doan_on_dinh
     FROM '@torrents_clean.parquet' WHERE NOT implausible""")

,cap_torrent_phim_sach,so_torrent,so_phim,trong_giai_doan_on_dinh
0,2078927,2066598,126081,923998


## Tóm tắt quy trình

| Giai đoạn | Script | Đầu ra chính | Ý nghĩa |
|---|---|---|---|
| Trích xuất | 01–02, 10 | `matched_files`, `magnetdb_matched_slice`, `mdb_*` | Lấy phần cần dùng từ SQLite 117 GB |
| Chuẩn hóa | 03 | `imdb_title_index` | Mọi tên phim ở mọi ngôn ngữ về một dạng |
| Tích hợp | 04, 05, 06–07 | `name_matches`, `movies`, `tmdb_*` | Ghép torrent ↔ IMDb ↔ TMDb |
| Kiểm tra | Bước 8 notebook này | — | So thuật toán ghép tên với nhãn MagnetDB: mức đồng thuận, độ phủ |
| Làm sạch | 09 | `torrents_clean`, `movie_analysis` | 10 bộ lọc A–J, loại lỗi ghép và torrent giả |

Các notebook 01–08 dùng hai bảng sạch này để phân tích và vẽ biểu đồ.